# Outlier-corrupted MAD regression — Figures 3 and 4

This notebook accompanies Section 5.2 and Appendix E.4 of `IPM_DRO.pdf`.
The dimension sweep produces Figure 3, and the contamination-scale sweep produces Figure 4.
Start Jupyter from the repository root or `regression/`.

The original notebook includes an extra dimension `d=3`; the paper shows `d=1,5,10,50`.
The CLI in `run_regression.py` defaults to those four paper dimensions.
The first diagnostic cells use a separate lognormal setup and are not paper results.
See `../docs/regression_notes.md` for source/paper differences and lightweight commands.
Full sweeps are computationally expensive; saved results can be plotted with the CLI.


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import os, sys, importlib, warnings
from pathlib import Path
import numpy as np
import cvxpy as cp
import pandas as pd
import matplotlib.pyplot as plt
from math import ceil, floor
from concurrent.futures import ProcessPoolExecutor, as_completed

# Suppress CVXPY reshape order FutureWarning
warnings.filterwarnings(
    'ignore',
    message='.*order of the reshape expression.*',
    category=FutureWarning,
)

# Add worker directory to path so spawn workers can find it
PROJECT_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "regression" / "exp1_oos_worker.py").is_file()
)
WORKER_DIR = str(PROJECT_ROOT / "regression")
RESULTS_DIR = PROJECT_ROOT / "results" / "regression"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MAX_WORKERS = max(1, int(os.environ.get("IPM_DRO_WORKERS", min(4, os.cpu_count() or 1))))
if WORKER_DIR not in sys.path:
    sys.path.insert(0, WORKER_DIR)

print('imports ok')
from exp1_oos_worker import _solve


In [ ]:
# ── Inline solver functions (for unit tests; worker uses exp1_oos_worker.py) ──

def cheap_robust_mean(X, eps):
    n = X.shape[0]
    X_sorted = np.sort(X, axis=0)
    start = ceil(eps * n)
    end   = floor((1 - eps) * n)
    if start >= end:
        return X.mean(axis=0)
    return X_sorted[start:end].mean(axis=0)


def lad_regression(X, y, theta_bound=1.0):
    n, d = X.shape
    Z    = np.hstack([X, y.reshape(-1, 1)])
    theta = cp.Variable(d)
    s     = cp.Variable(n, nonneg=True)
    a     = cp.hstack([theta, np.array([-1.0])])
    prob  = cp.Problem(cp.Minimize(cp.mean(s)),
                       [s >= Z @ a, s >= -(Z @ a),
                        cp.norm(theta, 2) <= theta_bound])
    _solve(prob)
    return theta.value


def wdro_regression(X, y, rho, theta_bound=1.0):
    """Standard WDRO for regression (L1 loss, dual norm = 2)."""
    n, d = X.shape
    Z     = np.hstack([X, y.reshape(-1, 1)])
    theta = cp.Variable(d)
    s     = cp.Variable(n, nonneg=True)
    a     = cp.hstack([theta, np.array([-1.0])])
    prob  = cp.Problem(
        cp.Minimize(rho * cp.norm(a, 2) + cp.mean(s)),
        [s >= Z @ a, s >= -(Z @ a),
         cp.norm(theta, 2) <= theta_bound]
    )
    _solve(prob)
    return theta.value


def or_wdro_regression(X, y, sigma, rho, eps_tv, z_0, theta_bound=1.0):
    """
    OR-WDRO for regression (L1 loss, q=2, p=1).
    z_0 : robust center in augmented space R^{d+1} = [x; y].
    """
    n, d = X.shape
    D    = d + 1
    Z    = np.hstack([X, y.reshape(-1, 1)])
    z_0  = z_0.ravel()

    theta    = cp.Variable(d)
    lambda_1 = cp.Variable(nonneg=True)
    lambda_2 = cp.Variable(nonneg=True)
    alpha    = cp.Variable()
    s        = cp.Variable(n, nonneg=True)
    zG1 = cp.Variable((D, n)); zW1 = cp.Variable((D, n)); t1 = cp.Variable(n, nonneg=True)
    zG2 = cp.Variable((D, n)); zW2 = cp.Variable((D, n)); t2 = cp.Variable(n, nonneg=True)

    obj  = cp.Minimize(
        lambda_1 * sigma**2 + lambda_2 * rho
        + cp.sum(s) / (n * (1 - eps_tv)) + alpha
    )
    cons = []
    for i in range(n):
        g1i = zG1[:, i]; w1i = zW1[:, i]
        g2i = zG2[:, i]; w2i = zW2[:, i]
        # j=1 (s=+1): [-theta; 1] + zG1 + zW1 = 0
        cons.append(cp.hstack([-theta, np.array([1.0])]) + g1i + w1i == 0)
        cons.append(s[i] >= z_0 @ g1i + t1[i] + Z[i] @ w1i - alpha)
        cons.append(cp.SOC(lambda_1 + 0.5*t1[i],
            cp.hstack([2*g1i, cp.reshape(lambda_1 - 0.5*t1[i], (1,), order='F')])))
        cons.append(cp.norm(w1i, 2) <= lambda_2)
        # j=2 (s=-1): [theta; -1] + zG2 + zW2 = 0
        cons.append(cp.hstack([theta, np.array([-1.0])]) + g2i + w2i == 0)
        cons.append(s[i] >= z_0 @ g2i + t2[i] + Z[i] @ w2i - alpha)
        cons.append(cp.SOC(lambda_1 + 0.5*t2[i],
            cp.hstack([2*g2i, cp.reshape(lambda_1 - 0.5*t2[i], (1,), order='F')])))
        cons.append(cp.norm(w2i, 2) <= lambda_2)

    cons.append(cp.norm(theta, 2) <= theta_bound)
    prob = cp.Problem(obj, cons)
    _solve(prob)
    return theta.value


def mom_dro_regression(X, y, epsilon, K, M_samples, Omega, R=1.0, theta_bound=1.0):
    """
    MoM IPM-DRO for scalar regression (L1 loss, k=1, J=2).
    z_m sampled uniformly from ball of radius R in R^d.
    """
    n, d = X.shape
    M    = M_samples
    Xi   = np.hstack([X, y.reshape(-1, 1)])

    rng   = np.random.default_rng()
    Z_raw = rng.standard_normal((M, d))
    Z_dir = Z_raw / np.linalg.norm(Z_raw, axis=1, keepdims=True)      # uniform on sphere
    r     = rng.uniform(0, 1, size=(M, 1)) ** (1 / d) * R             # radial CDF: P(r<t) = (t/R)^d
    Z     = Z_dir * r                                                  # (M, d), uniform in ball

    am        = np.hstack([Z, -np.ones((M, 1))])   # (M, d+1)
    B         = n // K
    Xi_use    = Xi[:K * B]
    inner     = Xi_use @ am.T
    loss      = np.abs(inner)
    blk_means = loss.reshape(K, B, M).mean(axis=1)
    mu_mom    = blk_means[0] if K == 1 else np.median(blk_means, axis=0)
    tau       = mu_mom + epsilon

    theta = cp.Variable(d)
    alpha = cp.Variable(nonneg=True)
    beta  = cp.Variable(nonneg=True)
    nu    = cp.Variable(M, nonneg=True)
    zeta1 = cp.Variable(nonneg=True)
    zeta2 = cp.Variable(nonneg=True)
    lam1p = cp.Variable(M, nonneg=True)
    lam1n = cp.Variable(M, nonneg=True)
    lam2p = cp.Variable(M, nonneg=True)
    lam2n = cp.Variable(M, nonneg=True)

    diff1 = lam1p - lam1n
    diff2 = lam2p - lam2n
    c1 = cp.hstack([
        theta - (1.0/M)*(Z.T @ diff1),
        cp.reshape(-1.0 + (1.0/M)*cp.sum(diff1), (1,))
    ])
    c2 = cp.hstack([
        -theta - (1.0/M)*(Z.T @ diff2),
        cp.reshape(1.0 + (1.0/M)*cp.sum(diff2), (1,))
    ])

    cons = [
        zeta1 <= alpha, zeta2 <= alpha,
        lam1p + lam1n == nu,
        lam2p + lam2n == nu,
        cp.SOC(zeta1 + beta, cp.hstack([c1, cp.reshape(zeta1 - beta, (1,))])),
        cp.SOC(zeta2 + beta, cp.hstack([c2, cp.reshape(zeta2 - beta, (1,))])),
        cp.norm(theta, 2) <= theta_bound,
    ]
    prob = cp.Problem(cp.Minimize(alpha + beta * Omega + (1.0/M)*(nu @ tau)), cons)
    _solve(prob)
    th = theta.value
    return th if (th is not None and not np.any(np.isnan(th))) else None

print('solvers defined')

In [ ]:
# ── Synthetic unit test ────────────────────────────────────────────────────────
# Mirrors exp1_oos.m: lognormal X, L1 loss, adversarial contamination + y-shift.

import warnings
warnings.filterwarnings('ignore')

rng_u = np.random.default_rng(0)
d_u, n_u, n_test_u = 5, 40, 300
eps_u      = 0
rho_u      = 0.1
sigma_u    = float(np.sqrt(d_u)) * np.e
c_scale_u  = 5
M_u        = 50

theta_star_u = rng_u.standard_normal(d_u)
theta_star_u /= np.linalg.norm(theta_star_u)

X_te_u = np.exp(rng_u.standard_normal((n_test_u, d_u)))
y_te_u = X_te_u @ theta_star_u

X_u = np.exp(rng_u.standard_normal((n_u, d_u)))
y_u = X_u @ theta_star_u
n_cont_u = floor(eps_u * n_u)
X_u[:n_cont_u] *= c_scale_u
y_u[:n_cont_u] *= -(c_scale_u**2)
y_u += rho_u
perm_u = rng_u.permutation(n_u)
X_u = X_u[perm_u]; y_u = y_u[perm_u]

def oos(th):
    if th is None: return float('nan')
    return float(np.mean(np.abs(X_te_u @ th - y_te_u)))

z_0_u = cheap_robust_mean(np.hstack([X_u, y_u.reshape(-1,1)]), 2*eps_u)
Xi_u  = np.hstack([X_u, y_u.reshape(-1,1)])
Omega_u = float(np.mean(np.sum(Xi_u**2, axis=1)))

K_u   = max(3, min(floor(n_u/3), ceil(4*(1+2*eps_u)/(1-2*eps_u)**2 * np.log(10))))
R_u   = 1.8   # rough scale for lognormal data

th_lad = lad_regression(X_u, y_u)
th_std = wdro_regression(X_u, y_u, rho_u)
th_or  = or_wdro_regression(X_u, y_u, sigma_u, rho_u, eps_u, z_0_u)
for epl in [0, 0.1, 0.5, 0.9, 1.5, 2.0]:
    th_mom1 = mom_dro_regression(X_u, y_u, epl, K_u, M_u, Omega_u, R=R_u)
    print(f'MoM DRO:     {oos(th_mom1):.4f}  (epsilon={epl})')

# print(f'Oracle risk (theta_star): {oos(theta_star_u):.4f}  (should be ~0)')
print(f'Plain LAD:   {oos(th_lad):.4f}')
# print(f'std WDRO:    {oos(th_std):.4f}')
# print(f'OR-WDRO:     {oos(th_or):.4f}')
# print(f'MoM DRO:     {oos(th_mom1):.4f}  (K={K_u} R={R_u})')


warnings.resetwarnings()

In [ ]:
# ── Smoke test via ProcessPoolExecutor (1 worker, 1 task) ─────────────────────

import exp1_oos_worker as w1
importlib.reload(w1)
from exp1_oos_worker import run_one_trial

cfg_smoke = dict(
    d=5, n_test=200, eps=0.22, contamination_scale=5,
    rho_shift=0.1, sigma=float(np.sqrt(5)) * np.e,
    eps_tv=0.22, rho_list=[0.05, 0.1, 0.2, 0.5, 1.0],
    M_samples=200,
    seed=42, delta=0.1, split_rate=0.8,
)

with ProcessPoolExecutor(max_workers=1) as ex:
    fut = ex.submit(run_one_trial, (0, 20, cfg_smoke))
    res = fut.result()

print('Smoke test result:', res)

In [ ]:
# ── Dimension sweep: d = 3, 5, 10, 20  (run & save) ──────────────────────────

import exp1_oos_worker as w1
importlib.reload(w1)
from exp1_oos_worker import run_one_trial

dims         = [1, 3, 5, 10, 50]
sample_sizes = [10, 20, 50, 75, 100]
T            = 50
n_test       = 1000
eps          = 0.2
save_dir_dim = str(RESULTS_DIR / 'dim_sweep')
os.makedirs(save_dir_dim, exist_ok=True)

for d_val in dims:
    path = os.path.join(save_dir_dim, f'dim_sweep_d{d_val}.csv')
    if os.path.exists(path):
        print(f'd={d_val}: already exists, skipping')
        continue

    cfg_d = dict(
        d=d_val, n_test=n_test, eps=eps, contamination_scale=10,
        rho_shift=0.1, sigma=float(np.sqrt(d_val) + 1) * np.e,
        eps_tv=eps, rho_list=[0.01, 0.1, 0.2, 0.5, 1.0],
        M_samples=800,
        seed=42, delta=0.1, split_rate=0.8,
    )
    tasks_d = [(t, n, cfg_d) for n in sample_sizes for t in range(T)]
    print(f'd={d_val}: submitting {len(tasks_d)} tasks ...')

    rows_d = []
    completed = 0
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = {ex.submit(run_one_trial, task): task for task in tasks_d}
        for fut in as_completed(futs):
            rows_d.append(fut.result())
            completed += 1
            if completed % len(sample_sizes) == 0:
                print(f'  progress: {completed}/{len(tasks_d)}')

    pd.DataFrame(rows_d).to_csv(path, index=False)
    print(f'd={d_val}: saved {len(rows_d)} rows to {path}')

print('All dims done.')


In [ ]:
# ── Dimension sweep: plot from saved CSVs ─────────────────────────────────────

save_dir_dim = str(RESULTS_DIR / 'dim_sweep')
import re as _re
import glob as _glob
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sample_sizes = [10, 20, 50, 75, 100]
_csvs = sorted(_glob.glob(os.path.join(save_dir_dim, 'dim_sweep_d*.csv')))
dims  = sorted([int(_re.search(r'd(\d+)\.csv', os.path.basename(p)).group(1)) for p in _csvs])
print(f'Found dims: {dims}')

methods = ['risk_std', 'risk_or_2eps', 'risk_or_eps', 'risk_mom']
labels  = ['Std WDRO', 'OR-WDRO (2ω)', 'OR-WDRO (ω)', 'MoM TIPM-DRO']
colors  = ['red', 'green', 'limegreen', 'orange']

for d_val in dims:
    path = os.path.join(save_dir_dim, f'dim_sweep_d{d_val}.csv')
    df_d = pd.read_csv(path)

    means_d = {m: [] for m in methods}
    p10_d   = {m: [] for m in methods}
    p90_d   = {m: [] for m in methods}
    for n in sample_sizes:
        sub = df_d[df_d['n'] == n]
        for m in methods:
            vals = sub[m].dropna().values
            means_d[m].append(float(np.mean(vals)) if vals.size else np.nan)
            p10_d[m].append(float(np.percentile(vals, 10)) if vals.size else np.nan)
            p90_d[m].append(float(np.percentile(vals, 90)) if vals.size else np.nan)

    fig, ax = plt.subplots(figsize=(7, 5))
    for m, lbl, col in zip(methods, labels, colors):
        mn = np.array(means_d[m])
        lo = np.array(p10_d[m])
        hi = np.array(p90_d[m])
        ax.plot(sample_sizes, mn, color=col, marker='o', label=lbl)
        ax.fill_between(sample_sizes, lo, hi, color=col, alpha=0.15)

    #ax.set_title(f'OOS Excess Risk vs Training Size with 10-90% Quantile', fontsize=16)
    ax.set_xlabel('Training Sample Size $N$', fontsize=16)
    ax.set_ylabel('OOS excess risk (MAD)', fontsize=16)
    ax.tick_params(axis='both', labelsize=13)
    ax.legend(fontsize=14)
    ax.set_xlim(sample_sizes[0] - 2, sample_sizes[-1] + 5)
    plt.tight_layout()
    out_path = str(RESULTS_DIR / f'exp1_oos_dim_sweep_d{d_val}.pdf')
    plt.savefig(out_path)
    plt.show()
    print(f'Saved {out_path}')


In [ ]:
# ── Contamination scale sensitivity (n=50, vary contamination_scale) ──────────
# Fix n=50, T=50 trials. Sweep contamination_scale to see how each method
# degrades as outliers become more extreme.

import exp1_oos_worker as w1
importlib.reload(w1)
from exp1_oos_worker import run_one_trial

cont_scales  = [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
n_cont       = 50
T_cont       = 50

cfg_cont_base = dict(
    d=10, n_test=1000, eps=0.2,
    rho_shift=0.1, sigma=float(np.sqrt(10) + 1) * np.e,
    eps_tv=0.2, rho_list=[0.01, 0.05, 0.1, 0.5, 1.0, 2.0],
    M_samples=800,
    seed=25, delta=0.1, split_rate=0.8,
)

save_dir_cont = str(RESULTS_DIR / 'cont_scale')
os.makedirs(save_dir_cont, exist_ok=True)

for ci, cs in enumerate(cont_scales):
    out_path = os.path.join(save_dir_cont, f'cont_scale_cs{cs}.csv')
    if os.path.exists(out_path):
        print(f'cs={cs}: already exists, skipping')
        continue

    tasks_cs = [
        (t, n_cont, {**cfg_cont_base, 'contamination_scale': cs, 'seed': 42 + ci})
        for t in range(T_cont)
    ]
    print(f'cs={cs}: submitting {len(tasks_cs)} tasks ...')

    rows = []
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = {ex.submit(run_one_trial, task): task for task in tasks_cs}
        for fut in as_completed(futs):
            rows.append(fut.result())

    pd.DataFrame(rows).to_csv(out_path, index=False)
    print(f'cs={cs}: saved {len(rows)} rows to {out_path}')

print('All contamination scales done.')


In [ ]:
# ── Report ──
save_dir_cont = str(RESULTS_DIR / 'cont_scale')
import glob as _glob
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
_csvs = sorted(_glob.glob(os.path.join(save_dir_cont, 'cont_scale_cs*.csv')))
df_cont = pd.concat([pd.read_csv(p) for p in _csvs], ignore_index=True)
cont_scales = sorted(df_cont['contamination_scale'].unique().astype(int).tolist())
print(f'Loaded {len(df_cont)} rows, scales: {cont_scales}')

methods_c = [ 'risk_or_2eps', 'risk_or_eps', 'risk_mom']
labels_c  = ['OR-WDRO (2ω)', 'OR-WDRO (ω)', 'MoM TIPM-DRO']
colors_c  = ['green', 'limegreen', 'orange']

means_c = {m: [] for m in methods_c}
p10_c   = {m: [] for m in methods_c}
p90_c   = {m: [] for m in methods_c}

for cs in cont_scales:
    sub = df_cont[df_cont['contamination_scale'] == cs]
    for m in methods_c:
        vals = sub[m].dropna().values
        means_c[m].append(float(np.mean(vals)) if vals.size else np.nan)
        p10_c[m].append(float(np.percentile(vals, 10)) if vals.size else np.nan)
        p90_c[m].append(float(np.percentile(vals, 90)) if vals.size else np.nan)

print(f"\n{'':20s}" + "".join(f"  cs={cs:>3d}" for cs in cont_scales))
for m, lbl in zip(methods_c, labels_c):
    mean_row = "  ".join(f"{means_c[m][i]:6.3f}" for i in range(len(cont_scales)))
    p10_row  = "  ".join(f"{p10_c[m][i]:6.3f}"  for i in range(len(cont_scales)))
    p90_row  = "  ".join(f"{p90_c[m][i]:6.3f}"  for i in range(len(cont_scales)))
    print(f"\n{lbl}")
    print(f"  mean : {mean_row}")
    print(f"  p10  : {p10_row}")
    print(f"  p90  : {p90_row}")

# ── Plot ──
fig2, ax2 = plt.subplots(figsize=(8, 5))
for m, lbl, col in zip(methods_c, labels_c, colors_c):
    mn = np.array(means_c[m])
    lo = np.array(p10_c[m])
    hi = np.array(p90_c[m])
    ax2.plot(cont_scales, mn, color=col, marker='o', label=lbl)
    ax2.fill_between(cont_scales, lo, hi, color=col, alpha=0.15)

ax2.set_xlabel('Contamination scale', fontsize=16)
ax2.set_ylabel('OOS excess risk (MAD)', fontsize=16)
#ax2.set_title('OOS Excess Risk vs Contamination Scale with 10-90% Quantile', fontsize=16)
ax2.tick_params(axis='both', labelsize=13)
ax2.legend(fontsize=16)
ax2.set_xticks(cont_scales)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'exp1_contamination_scale.pdf')
plt.show()
